# Train your IrishQwen in Colab

Write the code in your fork, commit and push it, then come here to train it. This notebook runs **your implementation**; finish the classroom blanks first.

1. Save a copy of this notebook to your Drive.
2. Choose **Runtime → Change runtime type → T4 GPU**, then connect. Free GPU access depends on availability.
3. Enter your GitHub username below and run the cells in order.

Your fork must be public for this notebook's unauthenticated clone. The default repository name is `irishQwen`. Changes in github.dev must be committed and pushed before Colab can see them.


In [ ]:
# Your fork, not the instructor's copy!
GITHUB_USERNAME = "" # @param {type:"string"}
REPOSITORY = "irishQwen" # @param {type:"string"}
BRANCH = "master" # @param {type:"string"}
EPOCHS = 5 # @param {type:"integer"}


## Check the GPU
Our model automatically uses CUDA when it is available. Stop here if Colab has given you a CPU runtime.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU connected. Choose Runtime → Change runtime type → T4 GPU and reconnect.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)


## Get your code and dependencies
This makes a fresh checkout each time you run the cell. Earlier checkouts and their checkpoints stay in the runtime. To pick up new commits from your fork, rerun this cell and the cells below it.


In [ ]:
from pathlib import Path
import re
import subprocess
import sys
import tempfile

# Pipe the script's output through Python so Colab actually displays it.
def run(command, cwd=None):
    with subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9-]*", GITHUB_USERNAME):
    raise ValueError("Enter your GitHub username in the first cell.")
if not re.fullmatch(r"[A-Za-z0-9_.-]+", REPOSITORY) or REPOSITORY in (".", ".."):
    raise ValueError("Enter a repository name, not a URL.")
if not BRANCH or BRANCH.startswith("-"):
    raise ValueError("Enter the branch containing your class work.")

repo_dir = Path(tempfile.mkdtemp(prefix="irishqwen-", dir="/content")) / "repo"
url = f"https://github.com/{GITHUB_USERNAME}/{REPOSITORY}.git"
run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, str(repo_dir)])
run([sys.executable, "-m", "pip", "install", "-r", str(repo_dir / "requirements.txt")])
print("Working directory:", repo_dir)
run(["git", "log", "-1", "--format=%h %s"], cwd=repo_dir)


## Train on Shakespeare
The default is five epochs. For a quick implementation check, set `EPOCHS` to `1` in the first cell, then rerun that cell and this one. Other model settings use the training script's defaults.

Training and validation losses appear below as the script runs. After training, it loads the best checkpoint and prints sample passages in the same output. If you get an error in a layer, use the traceback to find the method that still needs work. Fix it in your fork, commit and push, then get a fresh checkout above.


In [ ]:
import time

if not isinstance(EPOCHS, int) or isinstance(EPOCHS, bool) or EPOCHS < 1:
    raise ValueError("EPOCHS must be a positive integer.")
checkpoint = repo_dir / "Datasets" / "qwen_shakespeare.pt"
started = time.perf_counter()
run([
    sys.executable, "-u", "train_qwen_shakespeare.py",
    "--optimizer", "adam", "--epochs", str(EPOCHS),
    "--save", str(checkpoint),
], cwd=repo_dir)
elapsed = time.perf_counter() - started
print(f"Total time (training, validation, and samples): {elapsed / 60:.2f} minutes")


## Download your trained model
Colab's runtime files are temporary. Download the checkpoint before disconnecting. It contains model weights and configuration; keep your fork and its tokenizer vocabulary too.


In [ ]:
from google.colab import files

if not checkpoint.is_file():
    raise FileNotFoundError("No checkpoint yet. Complete a training run first.")
files.download(str(checkpoint))


When you're done, choose **Runtime → Disconnect and delete runtime** to release the GPU. Your committed code stays on GitHub and your notebook copy stays in Drive; files in the runtime do not.
